In [1]:
import os
import random
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# === Load your chunked document CSV ===
csv_path = r"C:\Users\japje\Documents\B.G\wiki_hybrid_chunks_300.csv"
df = pd.read_csv(csv_path)
texts = df["chunk_text"].tolist()
id_to_text = df[["chunk_id", "chunk_text"]].reset_index(drop=True)

# === Load embedding model and encode chunks ===
print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding document chunks...")
embeddings = embed_model.encode(texts, convert_to_numpy=True, batch_size=64, show_progress_bar=True)
dim = embeddings.shape[1]

# === Build FAISS index ===
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

# === Load TinyLLaMA model ===
print("Loading TinyLLaMA model...")
tinyllama_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(tinyllama_model_id)
model = AutoModelForCausalLM.from_pretrained(tinyllama_model_id)
llama_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=-1)

# === Ask question function ===
def ask_question(query, k=5):
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, k)
    retrieved_chunks = [id_to_text.iloc[idx]["chunk_text"] for idx in indices[0]]
    context = "\n".join(retrieved_chunks[:3])
    if len(context) > 1000:
        context = context[:1000]

    prompt = f"""You are a helpful assistant. Use the following context to answer the question.

Context:
{context}

Question: {query}
Answer:"""

    response = llama_pipe(prompt, max_new_tokens=512, temperature=0.7, do_sample=True)
    full_response = response[0]["generated_text"]

    if "Answer:" in full_response:
        answer = full_response.split("Answer:")[-1].strip()
    else:
        answer = full_response.strip()

    return prompt, answer

# === Feedback CSV path and initializer ===
feedback_csv_path = "feedback_log.csv2"
if not os.path.exists(feedback_csv_path):
    pd.DataFrame(columns=["prompt", "response", "feedback"]).to_csv(feedback_csv_path, index=False)

# === Save feedback to CSV function ===
def save_feedback(prompt, answer, feedback_type):
    new_row = pd.DataFrame([{
        "prompt": prompt,
        "response": answer,
        "feedback": feedback_type
    }])
    new_row.to_csv(feedback_csv_path, mode='a', header=False, index=False)

# === Auto-ask questions and collect feedback ===
def auto_ask_and_collect_feedback(questions_df):
    for i, row in questions_df.iterrows():
        query = row["question"]
        prompt, answer = ask_question(query)
        feedback_type = random.choice(["thumbs_up", "thumbs_down"])  # or you can use your own logic here
        save_feedback(prompt, answer, feedback_type)
        print(f"Q{i+1}: {query}")
        print(f"A: {answer}")
        print(f"Feedback: {feedback_type} saved.\n")

# === Load your question CSV ===
questions_df = pd.read_csv("generated_questions_tinyllama.csv2")  # change to your CSV file path

# === Run the automatic Q&A with feedback collection ===
auto_ask_and_collect_feedback(questions_df)
print(f"Feedback saved in {feedback_csv_path}")


Loading embedding model...
Encoding document chunks...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Loading TinyLLaMA model...


Device set to use cpu


Q1: Who provides the voice of the animated lead characters in the TV series Delilah julius and Pearlie?
A: Delilah julius and Pearlie are voiced by the same person.
Feedback: thumbs_up saved.

Q2: How did the filmography of Marieve Herington include two more albums and two theme songs for television programs?
A: In the filmography of Marieve Herington, the two more albums she appeared on were 2021's Midnight Sessions and 2010's E Midnight Sessions. She also performed the theme songs for the TV shows Sesame Park, Pippi Longstocking, and Marigold's Mathemagics.
Feedback: thumbs_down saved.

Q3: How do you describe Honoka's character as a force of nature?
A: Honoka is a force of nature, a creature born of the elements that is beyond our understanding and interconnected with the world around her. She is a creature with a powerful will and a fierce determination to stand up for what is right, no matter the cost. Honoka is a force that cannot be tamed, and she is always ready to use her powe